# US Bureau Census

This notebook is a testing of the how we will grab data from US Bureau Census for population data

In [1]:
import duckdb

import requests
import pandas as pd
import os
from dotenv import load_dotenv

from pipeline.dbloader import DBLoader

In [2]:
load_dotenv()
FRED_KEY = os.getenv("FRED_API_KEY")

Using: Federal Reserve Economic Detail

In [3]:
import requests
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()
FRED_KEY = os.getenv("FRED_API_KEY")

def fetch_fred_population(start_year: int = 1950, end_year: int = 2025) -> pd.DataFrame:
    """
    Fetches annual US population from FRED series POPTHM (BEA, monthly).
    Aggregates monthly to annual by averaging — same methodology BEA uses
    for its annual series. Units are in thousands.
    """
    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {
        "series_id":        "POPTHM",
        "api_key":          FRED_KEY,
        "file_type":        "json",
        "observation_start": f"{start_year}-01-01",
        "observation_end":   f"{end_year}-12-31",
        "frequency":        "a",        # aggregate to annual
        "aggregation_method": "avg",    # annual average of monthly estimates
        "units":            "lin",
    }

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()

    observations = r.json()["observations"]
    df = pd.DataFrame(observations)[["date", "value"]]
    df = (
        df
        .assign(
            year       = pd.to_datetime(df["date"]).dt.year,
            population = pd.to_numeric(df["value"], errors="coerce") * 1000  # convert thousands to actual
        )
        .dropna(subset=["population"])
        .astype({"population": int})
        [["year", "population"]]
        .reset_index(drop=True)
    )
    return df

df_pop = fetch_fred_population()
print(df_pop)

    year  population
0   1959   177130000
1   1960   180760000
2   1961   183742000
3   1962   186591000
4   1963   189300000
..   ...         ...
62  2021   332503000
63  2022   334350000
64  2023   337087000
65  2024   340095000
66  2025   341945000

[67 rows x 2 columns]


In [4]:
df_pop = fetch_fred_population()
pop_rel = duckdb.sql("SELECT * FROM df_pop")

In [5]:
# Write to DB

dbloader = DBLoader()
dbloader.write_to_postgres(
    relation=pop_rel,
    table_name="t_us_population",
    schema={
        "year":             ("NUMERIC", True),
        "population":       ("NUMERIC", False)
    }
)

Schema 'bronze' is ready.
[INFO] Created 'bronze.t_us_population' with PKs: ['year']
[INFO] Upserted 67 records into 'bronze.t_us_population' on PK conflict of ['year'].
